In [1]:
from google.colab import files
uploaded = files.upload()


import pandas as pd
import numpy as np

# Load datasets (no header usually in these files)
cleveland = pd.read_csv("Cleavland.csv", header=None)
hungary = pd.read_csv("hung.csv", header=None)
switzerland = pd.read_csv("Switzerland.csv", header=None)

# Assign column names (UCI Heart Dataset standard)
columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','target']

cleveland.columns = columns
hungary.columns = columns
switzerland.columns = columns

# Merge datasets
df = pd.concat([cleveland, hungary, switzerland], ignore_index=True)

print("Merged Dataset Shape:", df.shape)

Saving Switzerland.csv to Switzerland.csv
Saving hung.csv to hung.csv
Saving Cleavland.csv to Cleavland.csv
Merged Dataset Shape: (720, 14)


In [2]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors='coerce')

# Remove missing values
df.dropna(inplace=True)

# Remove negative values
df = df[(df >= 0).all(axis=1)]

print("After Cleaning:", df.shape)

After Cleaning: (298, 14)


In [5]:
# IQR method
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After Outlier Removal:", df.shape)

After Outlier Removal: (183, 14)


In [6]:
from sklearn.preprocessing import StandardScaler

# Convert target to binary (0 = no disease, 1 = disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# Split features and label
X = df.drop('target', axis=1)
y = df['target']

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [8]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", acc_lr)

Logistic Regression Accuracy: 0.7297297297297297


In [9]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)

print("kNN Accuracy:", acc_knn)

kNN Accuracy: 0.7297297297297297


In [10]:
print("\nFinal Comparison:")
print("Logistic Regression:", acc_lr)
print("kNN:", acc_knn)

if acc_lr > acc_knn:
    print("✅ Logistic Regression is better")
else:
    print("✅ kNN is better")


Final Comparison:
Logistic Regression: 0.7297297297297297
kNN: 0.7297297297297297
✅ kNN is better
